# Week 41, Day 3: Internal models: knowing where your hand is

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w041/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Dead reckoning drifts

Write forward_only(v, gain_model): starting at the true initial position, add dt times gain_model times each commanded velocity, returning the estimated positions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(1995)
dt, n = 0.005, 600
t = np.arange(n) * dt
v_cmd = 0.3 * np.sin(2 * np.pi * t / 1.5) + 0.15                   # commanded velocity (m/s), known to the brain
gain_true, gain_model = 1.0, 0.9
x_true = np.zeros(n)
for i in range(1, n):
    x_true[i] = x_true[i - 1] + dt * gain_true * v_cmd[i - 1] + 0.0003 * r.normal()

def forward_only(v, gain_model):
    est = np.zeros(len(v))
    for i in range(1, len(v)):
        est[i] = ...
    return est

est_fwd = forward_only(v_cmd, gain_model)
err = np.abs(est_fwd - x_true)
print(f"forward model error: {1000 * err[100]:.1f} mm at 0.5 s, {1000 * err[-1]:.1f} mm at 3 s")
plt.plot(t, x_true, "k", label="true"); plt.plot(t, est_fwd, label="forward model only"); plt.xlabel("s"); plt.ylabel("position (m)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_drift():
    "the forward-model error in the last second is at least 1.5 times its error in the first second"
    err = np.abs(est_fwd - x_true)
    assert err[-200:].mean() > 1.5 * err[:200].mean()

test_drift()
print("All checks passed")

### Exercise: Delayed feedback lags

Build the delayed, noisy measurement y[i] = x_true[i - d] + noise (standard deviation 5 mm; for the first d steps, use x_true[0]), and use it directly as the estimate.

In [ ]:
d = 20
y = ...
est_fb = y.copy()
err_fb = np.abs(est_fb - x_true)
print(f"feedback-only RMS error {1000 * np.sqrt(np.mean(err_fb ** 2)):.1f} mm")
plt.plot(t, x_true, "k", label="true"); plt.plot(t, est_fb, label="delayed feedback only", alpha=0.7); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_lag():
    "the feedback-only error is larger while the hand moves fast than while it moves slowly"
    speed = np.abs(np.gradient(x_true, dt))
    err = np.abs(est_fb - x_true)
    assert err[speed > np.percentile(speed, 75)].mean() > 1.5 * err[speed < np.percentile(speed, 25)].mean()

test_lag()
print("All checks passed")

### Exercise: Prediction plus correction

Complete the loop of combine(y, v, K=0.1): update the delayed estimate xd with the measurement y[i] (after first predicting it forward one step with the forward model), then set the current estimate to xd plus the forward model's displacement over the last d commands.

In [ ]:
def combine(y, v, K=0.1):
    est = np.zeros(n); xd = 0.0
    for i in range(1, n):
        if i > d:
            xd = xd + dt * gain_model * v[i - d - 1]                 # forward model predicts x[i - d]
            xd = ...                                                  # correct with the delayed measurement
        recent = v[max(0, i - d):i]
        est[i] = ...
    return est

est_comb = combine(y, v_cmd)
rms = {name: 1000 * np.sqrt(np.mean((e - x_true) ** 2)) for name, e in (("forward", est_fwd), ("feedback", est_fb), ("combined", est_comb))}
print({k: f"{v:.1f} mm" for k, v in rms.items()}, "RMS error")
plt.plot(t, np.abs(est_fwd - x_true) * 1000, label="forward only"); plt.plot(t, np.abs(est_fb - x_true) * 1000, label="feedback only", alpha=0.6)
plt.plot(t, np.abs(est_comb - x_true) * 1000, label="combined"); plt.ylabel("error (mm)"); plt.xlabel("s"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_best():
    "the combined estimator's RMS error is lower than both alternatives"
    assert rms["combined"] < rms["forward"] and rms["combined"] < rms["feedback"]

def test_bounded():
    "the combined error does not grow: its last second is no worse than 1.5 times its second second"
    e = np.abs(est_comb - x_true)
    assert e[-200:].mean() < 1.5 * e[200:400].mean()

test_best()
test_bounded()
print("All checks passed")